# Product Confirmation Workflow

This notebook downloads DIST-ALERT products from S3, unzips them, and runs the confirmation workflow.

In [1]:
import pandas as pd
import shutil
import zipfile
import requests
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from dist_s1 import run_sequential_confirmation_of_dist_products_workflow
from utils import unzip_dist_s1_prod, wrap_run_sequential_confirmation_of_dist_products_workflow
import multiprocessing
from functools import partial


/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
token = 'dist-event-10max-v1_32'

In [3]:
tmp_dir =  Path(f'tmp_{token}')
unconfirmed_products_dir =  Path(f'unconfirmed_products_{token}')
confirmed_products_dir =  Path(f'confirmed_products_{token}')

tmp_dir.mkdir(exist_ok=True)
unconfirmed_products_dir.mkdir(exist_ok=True)
confirmed_products_dir.mkdir(exist_ok=True)

In [4]:
# Load the test products CSV
csv_path = Path('dist-s1-events_v1_32_10max.csv')
df = pd.read_csv(csv_path)
df.head()

,job_name,zip_url,s3_uri,browse_url,product_request_time,processing_duration,high_confidence_alert_threshold,mgrs_tile_id,post_date_buffer_days,stride_for_norm_param_estimation,...,memory_strategy,batch_size_for_norm_param_estimation,model_context_length,post_date,track_number,low_confidence_alert_threshold,n_workers_for_despeckling,device,max_pre_imgs_per_burst_mw,model_compilation
0,brazzaville_flood_and_landslides_2023__2023-12...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,947.558,4.5,33MWR,1,23,...,high,32,10,2024-02-11,109,2.5,4,best,none,False
1,brazzaville_flood_and_landslides_2023__2023-12...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,742.226,4.5,33MWR,1,23,...,high,32,10,2024-01-18,109,2.5,4,best,none,False
2,monkey_creek_fire_2024__2024-07-18__10TGQ__tra...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,710.178,4.5,10TGQ,1,23,...,high,32,10,2024-08-28,42,2.5,4,best,none,False
3,monkey_creek_fire_2024__2024-07-18__10TGQ__tra...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,575.377,4.5,10TGQ,1,23,...,high,32,10,2024-07-28,115,2.5,4,best,none,False
4,monkey_creek_fire_2024__2024-07-18__11TLK__tra...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,1046.205,4.5,11TLK,1,23,...,high,32,10,2024-07-25,64,2.5,4,best,none,False


In [5]:
def download_file(url, destination_path):
    dst_dir = destination_path.parent
    dst_dir.mkdir(exist_ok=True, parents=True)

    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(destination_path, 'wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                file.write(chunk)
    
    return destination_path


In [6]:
download_tasks = []
for _, row in df.iterrows():
    url = row['zip_url']
    filename = Path(url).name
    
    # Customize!!!!
    job_name = row['job_name']
    dist_event = job_name.split('__')[0]
    dst_dir = tmp_dir / dist_event
    zip_path = dst_dir / filename

    download_tasks.append((url, zip_path))

In [7]:
download_file_p = lambda t: download_file(*t)
with ThreadPoolExecutor(max_workers=10) as executor:
    paths = list(tqdm(executor.map(download_file_p, download_tasks[:]), total=len(download_tasks)))

100%|████████| 854/854 [1:34:06<00:00,  6.61s/it]


# Unzip

In [8]:
downloaded_zips = list(tmp_dir.rglob('*.zip'))
len(downloaded_zips)

854

In [9]:
num_processes = 8 
print('Total processes: ', multiprocessing.cpu_count())
print(f"Using {num_processes} processes for unzipping.")


with multiprocessing.Pool(processes=num_processes) as pool:
    unzipper = partial(unzip_dist_s1_prod, unconfirmed_products_dir=unconfirmed_products_dir)

    results = pool.imap(unzipper, downloaded_zips[:])
    for _ in tqdm(results, total=len(downloaded_zips), desc="Unzipping Files"):
        pass

Total processes:  12
Using 8 processes for unzipping.


Unzipping Files: 100%|█| 854/854 [00:47<00:00, 18


In [10]:
subdirs = list(unconfirmed_products_dir.rglob('OPERA_L3_DIST-ALERT-S1*/'))
mgrs_tiles_unzipped = list(set([subdir.parent.name for subdir in subdirs]))
mgrs_tiles_unzipped[:3]

['11TLK', '49MDN', '19HBD']

In [11]:
subdirs = list(unconfirmed_products_dir.rglob('OPERA_L3_DIST-ALERT-S1*/'))
mgrs_ts_unzipped_dirs = list(set([subdir.parent for subdir in subdirs]))
mgrs_ts_unzipped_dirs[:3]

[PosixPath('unconfirmed_products_dist-event-10max-v1_32/smokehouse_creek_fire_2024/14SME'),
 PosixPath('unconfirmed_products_dist-event-10max-v1_32/monkey_creek_fire_2024/10TGQ'),
 PosixPath('unconfirmed_products_dist-event-10max-v1_32/smokehouse_creek_fire_2024/14SLE')]

In [12]:
# cleanup_temp = True
# if cleanup_temp:
#     shutil.rmtree(tmp_dir)

# Confirmation

In [13]:
# confirm_kwargs = {'alert_low_conf_thresh': 3, 
#                   'alert_high_conf_thresh': 5, 
#                   'percent_reset_thresh': 50, 
#                   'no_day_limit': 18, 
#                   'no_count_reset_thresh': 4}

In [14]:
# confirmer = partial(wrap_run_sequential_confirmation_of_dist_products_workflow, 
#                         unconfirmed_products_dir=unconfirmed_products_dir, 
#                         confirmed_products_dir=confirmed_products_dir,
#                         confirm_kwargs=confirm_kwargs
#                        )
# list(map(confirmer, mgrs_ts_unzipped_dirs_f[:1]))

In [15]:
with multiprocessing.Pool(processes=5) as pool:
    confirmer = partial(wrap_run_sequential_confirmation_of_dist_products_workflow, 
                        unconfirmed_products_dir=unconfirmed_products_dir, 
                        confirmed_products_dir=confirmed_products_dir,
                        #confirm_kwargs=confirm_kwargs
                       )

    results = pool.imap(confirmer, mgrs_ts_unzipped_dirs[:])
    for _ in tqdm(results, total=len(mgrs_ts_unzipped_dirs), desc="Confirming Products"):
        pass

Confirming 14 products: 100%|█| 14/14 [01:35<00:s
Confirming 23 products: 100%|█| 23/23 [02:53<00:
Confirming 22 products: 100%|█| 22/22 [03:22<00:
Confirming 29 products: 100%|█| 29/29 [03:35<00:,
Confirming 22 products: 100%|█| 22/22 [03:57<00:9
Confirming 29 products: 100%|█| 29/29 [05:00<00:6
Confirming 23 products: 100%|█| 23/23 [03:44<00:5
Confirming 3 products: 100%|█| 3/3 [00:25<00:00,4
Confirming 20 products: 100%|█| 20/20 [03:17<00:
Confirming 21 products: 100%|█| 21/21 [03:55<00:
Confirming 30 products: 100%|█| 30/30 [04:06<00:4
Confirming 30 products: 100%|█| 30/30 [04:06<00:3
Confirming 26 products: 100%|█| 26/26 [04:27<00:
Confirming 26 products: 100%|█| 26/26 [05:17<00:
Confirming 32 products: 100%|█| 32/32 [05:37<00: 
Confirming 29 products: 100%|█| 29/29 [05:47<00: 
Confirming 14 products: 100%|█| 14/14 [02:32<00: 
Confirming 15 products: 100%|█| 15/15 [02:47<00: 
Confirming 24 products: 100%|█| 24/24 [04:02<00:
Confirming 23 products: 100%|█| 23/23 [04:15<00: 
Confirm